In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# SemEval Subtask 1 — XLM-R LARGE (Memory Safe + FP16)
# ============================================================

import os, glob, random
import torch
import numpy as np
import pandas as pdc
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm import tqdm
from torch.amp import autocast, GradScaler

# ---------- CONFIG ----------
MODEL_NAME = "xlm-roberta-large"
BASE_DIR = "/kaggle/input/semeval-data/subtask1"
TRAIN_DIR = f"{BASE_DIR}/train"
DEV_DIR = f"{BASE_DIR}/dev"
PRED_DIR = "/kaggle/working/predictions"
os.makedirs(PRED_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_LEN = 128
BATCH_SIZE = 2
ACCUM_STEPS = 8
EPOCHS = 3
LR = 8e-6
TEMPERATURE = 0.07
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---------- LANG CLUSTERS ----------
LANG_CLUSTERS = {
    "western": ["eng","deu","ita","spa","pol"],
    "indic": ["hin","ben","tel","ori","pan","urd","nep"],
    "semitic": ["arb","fas"],
    "african": ["amh","hau","swa"],
    "southeast_asia": ["mya","khm"],
    "sinitic": ["zho"],
    "turkic": ["tur"],
    "slavic": ["rus"],
}
LANG2CLUSTER = {l:c for c,ls in LANG_CLUSTERS.items() for l in ls}
CLUSTER2ID = {c:i for i,c in enumerate(LANG_CLUSTERS)}

# ---------- DATA ----------
def load_train():
    dfs=[]
    for f in glob.glob(f"{TRAIN_DIR}/*.csv"):
        lang=os.path.basename(f).split(".")[0]
        df=pd.read_csv(f)
        df["lang"]=lang
        dfs.append(df)
    df=pd.concat(dfs)
    strat=df.polarization.astype(str)+"_"+df.lang
    return train_test_split(df,test_size=0.15,stratify=strat,random_state=SEED)

def load_dev():
    data={}
    for f in glob.glob(f"{DEV_DIR}/*.csv"):
        lang=os.path.basename(f).split(".")[0]
        df=pd.read_csv(f)
        df["lang"]=lang
        data[lang]=df
    return data

class Dataset(torch.utils.data.Dataset):
    def __init__(self,df):
        self.t=df.text.tolist()
        self.l=df.polarization.tolist()
        self.lang=df.lang.tolist()
    def __len__(self): return len(self.t)
    def __getitem__(self,i): return self.t[i],self.l[i],self.lang[i]

# ---------- MODEL ----------
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=AutoModel.from_pretrained(MODEL_NAME)
        self.enc.gradient_checkpointing_enable()
        self.cluster=nn.Embedding(len(CLUSTER2ID),32)
        self.proj=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Linear(256,128))
        self.clf=nn.Linear(1024+32,2)

    def mean_pool(self,x,m):
        m=m.unsqueeze(-1)
        return (x*m).sum(1)/m.sum(1).clamp(min=1e-9)

    def forward(self,ids,mask,cid):
        out=self.enc(ids,mask).last_hidden_state
        emb=self.mean_pool(out,mask)
        return self.clf(torch.cat([emb,self.cluster(cid)],1)), self.proj(emb)

# ---------- LOSS ----------
class SupCon(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, z, y):
        z = nn.functional.normalize(z, dim=1)
        sim = torch.matmul(z, z.T) / self.temperature

        labels = y.unsqueeze(1)
        mask = (labels == labels.T).float().to(z.device)

        logits_mask = torch.ones_like(mask) - torch.eye(mask.size(0)).to(z.device)
        mask = mask * logits_mask

        exp_sim = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-9)

        loss = -(mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-9)
        return loss.mean()

# ---------- COLLATE ----------
def collate(b,tk):
    t,l,lang=zip(*b)
    cid=torch.tensor([CLUSTER2ID[LANG2CLUSTER[x]] for x in lang])
    enc=tk(list(t),padding=True,truncation=True,max_length=MAX_LEN,return_tensors="pt")
    return enc["input_ids"],enc["attention_mask"],torch.tensor(l),cid

# ---------- TRAIN ----------
def train():
    train_df,val_df=load_train()
    tk=AutoTokenizer.from_pretrained(MODEL_NAME)
    tl=DataLoader(Dataset(train_df),batch_size=BATCH_SIZE,shuffle=True,collate_fn=lambda x:collate(x,tk))
    vl=DataLoader(Dataset(val_df),batch_size=8,collate_fn=lambda x:collate(x,tk))

    model=Model().to(DEVICE)
    con=SupCon()
    opt=torch.optim.AdamW(model.parameters(),lr=LR)
    sch=get_linear_schedule_with_warmup(opt,200,EPOCHS*len(tl))
    scaler=GradScaler(device="cuda")

    for e in range(EPOCHS):
        model.train(); opt.zero_grad()
        for i,(ids,mask,y,c) in enumerate(tqdm(tl)):
            ids,mask,y,c=ids.to(DEVICE),mask.to(DEVICE),y.to(DEVICE),c.to(DEVICE)
            with autocast(device_type="cuda"):
                logits,z=model(ids,mask,c)
                loss=(nn.functional.cross_entropy(logits,y)+0.2*con(z,y))/ACCUM_STEPS
            scaler.scale(loss).backward()
            if (i+1)%ACCUM_STEPS==0:
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad()

        # validation
        model.eval(); yt,yp=[],[]
        for ids,mask,y,c in vl:
            ids,mask,c=ids.to(DEVICE),mask.to(DEVICE),c.to(DEVICE)
            logits,_=model(ids,mask,c)
            yp+=torch.argmax(logits,1).cpu().tolist()
            yt+=y.tolist()
        print(f"\n🔥 Epoch {e+1} Macro-F1:",f1_score(yt,yp,average="macro"))

    val_df = val_df.reset_index(drop=True)
    val_df["prob"] = probs   # ← USE SOFTMAX PROBABILITIES
    return model,tk,val_df

# ---------- THRESHOLDS ----------
def tune(df):
    th={}
    for l in df.lang.unique():
        sub=df[df.lang==l]
        best=0.5; bf=0
        for t in np.arange(0.2,0.8,0.02):
            f=f1_score(sub.polarization,(sub.prob>t).astype(int))
            if f>bf: bf=f; best=t
        th[l]=best
    return th

# ---------- PREDICT ----------
def predict(model,tk,th):
    dev=load_dev(); model.eval()
    for lang,df in dev.items():
        preds=[]
        for txt in tqdm(df.text,desc=lang):
            enc=tk(txt,return_tensors="pt",truncation=True,max_length=MAX_LEN).to(DEVICE)
            cid=torch.tensor([CLUSTER2ID[LANG2CLUSTER[lang]]]).to(DEVICE)
            logits,_=model(enc["input_ids"],enc["attention_mask"],cid)
            prob=torch.softmax(logits,1)[0,1].item()
            preds.append(int(prob>th.get(lang,0.5)))
        pd.DataFrame({"id":df.id,"polarization":preds}).to_csv(f"{PRED_DIR}/pred_{lang}.csv",index=False)

# ---------- RUN ----------
model,tk,val=train()
th=tune(val)
predict(model,tk,th)


In [ ]:
# ============================================================
# SemEval Subtask 1 — FINAL FIXED XLM-R LARGE PIPELINE
# ============================================================

import os, glob, random
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm import tqdm
from torch.amp import autocast, GradScaler

# ---------- CONFIG ----------
MODEL_NAME = "xlm-roberta-large"
BASE_DIR = "/kaggle/input/semeval/subtask1"
TRAIN_DIR = f"{BASE_DIR}/train"
DEV_DIR = f"{BASE_DIR}/dev"
PRED_DIR = "/kaggle/working/predictions"
os.makedirs(PRED_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_LEN = 160
BATCH_SIZE = 2
ACCUM_STEPS = 8
EPOCHS = 3
LR = 8e-6
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---------- LANG CLUSTERS ----------
LANG_CLUSTERS = {
    "western": ["eng","deu","ita","spa","pol"],
    "indic": ["hin","ben","tel","ori","pan","urd","nep"],
    "semitic": ["arb","fas"],
    "african": ["amh","hau","swa"],
    "southeast_asia": ["mya","khm"],
    "sinitic": ["zho"],
    "turkic": ["tur"],
    "slavic": ["rus"],
}
LANG2CLUSTER = {l:c for c,ls in LANG_CLUSTERS.items() for l in ls}
CLUSTER2ID = {c:i for i,c in enumerate(LANG_CLUSTERS)}

# ---------- DATA ----------
def load_train():
    dfs=[]
    for f in glob.glob(f"{TRAIN_DIR}/*.csv"):
        lang=os.path.basename(f).split(".")[0]
        df=pd.read_csv(f)
        df["lang"]=lang
        dfs.append(df)
    df=pd.concat(dfs)
    strat=df.polarization.astype(str)+"_"+df.lang
    return train_test_split(df,test_size=0.15,stratify=strat,random_state=SEED)

def load_dev():
    data={}
    for f in glob.glob(f"{DEV_DIR}/*.csv"):
        lang=os.path.basename(f).split(".")[0]
        df=pd.read_csv(f)
        df["lang"]=lang
        data[lang]=df
    return data

class Dataset(torch.utils.data.Dataset):
    def __init__(self,df):
        self.t=df.text.tolist()
        self.l=df.polarization.tolist()
        self.lang=df.lang.tolist()
    def __len__(self): return len(self.t)
    def __getitem__(self,i): return self.t[i],self.l[i],self.lang[i]

# ---------- MODEL ----------
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=AutoModel.from_pretrained(MODEL_NAME)
        self.enc.gradient_checkpointing_enable()
        self.cluster=nn.Embedding(len(CLUSTER2ID),32)
        self.proj=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Linear(256,128))
        self.clf=nn.Linear(1024+32,2)

    def mean_pool(self,x,m):
        m=m.unsqueeze(-1)
        return (x*m).sum(1)/m.sum(1).clamp(min=1e-9)

    def forward(self,ids,mask,cid):
        out=self.enc(ids,mask).last_hidden_state
        emb=self.mean_pool(out,mask)
        return self.clf(torch.cat([emb,self.cluster(cid)],1)), self.proj(emb)

# ---------- SUPCON ----------
class SupCon(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, z, y):
        z = nn.functional.normalize(z, dim=1)
        sim = torch.matmul(z, z.T) / self.temperature

        labels = y.unsqueeze(1)
        mask = (labels == labels.T).float().to(z.device)

        logits_mask = torch.ones_like(mask) - torch.eye(mask.size(0)).to(z.device)
        mask = mask * logits_mask

        exp_sim = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-9)

        loss = -(mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-9)
        return loss.mean()

# ---------- COLLATE ----------
def collate(b,tk):
    t,l,lang=zip(*b)
    cid=torch.tensor([CLUSTER2ID[LANG2CLUSTER[x]] for x in lang])
    enc=tk(list(t),padding=True,truncation=True,max_length=MAX_LEN,return_tensors="pt")
    return enc["input_ids"],enc["attention_mask"],torch.tensor(l),cid

# ---------- TRAIN ----------
def train():
    train_df,val_df=load_train()
    tk=AutoTokenizer.from_pretrained(MODEL_NAME)
    tl=DataLoader(Dataset(train_df),batch_size=BATCH_SIZE,shuffle=True,collate_fn=lambda x:collate(x,tk))
    vl=DataLoader(Dataset(val_df),batch_size=8,collate_fn=lambda x:collate(x,tk))

    model=Model().to(DEVICE)
    con=SupCon()
    opt=torch.optim.AdamW(model.parameters(),lr=LR)
    sch=get_linear_schedule_with_warmup(opt,200,EPOCHS*len(tl))
    scaler=GradScaler(device="cuda")

    for e in range(EPOCHS):
        model.train(); opt.zero_grad()
        for i,(ids,mask,y,c) in enumerate(tqdm(tl)):
            ids,mask,y,c=ids.to(DEVICE),mask.to(DEVICE),y.to(DEVICE),c.to(DEVICE)
            with autocast(device_type="cuda"):
                logits,z=model(ids,mask,c)
                loss=(nn.functional.cross_entropy(logits,y)+0.2*con(z,y))/ACCUM_STEPS
            scaler.scale(loss).backward()
            if (i+1)%ACCUM_STEPS==0:
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad()

        # validation
        model.eval(); yt,yp,probs=[],[],[]
        for ids,mask,y,c in vl:
            ids,mask,c=ids.to(DEVICE),mask.to(DEVICE),c.to(DEVICE)
            logits,_=model(ids,mask,c)
            prob=torch.softmax(logits,1)[:,1].cpu().tolist()
            yp+=prob
            yt+=y.tolist()

        print(f"\n🔥 Epoch {e+1} Macro-F1:",f1_score(yt,(np.array(yp)>0.5).astype(int),average="macro"))

    val_df = val_df.reset_index(drop=True)
    val_df["prob"] = yp
    return model,tk,val_df

# ---------- NORMALIZE PROBS ----------
def normalize_probs(val):
    val["prob"] = val.groupby("lang")["prob"].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-9)
    )
    return val

# ---------- CLUSTER THRESHOLDS ----------
def tune_cluster_thresholds(val):
    val["cluster"]=val.lang.map(lambda x:LANG2CLUSTER[x])
    th={}
    for c in val.cluster.unique():
        best,bf=0,0
        sub=val[val.cluster==c]
        for t in np.arange(-2,2,0.05):
            f=f1_score(sub.polarization,(sub.prob>t).astype(int))
            if f>bf: bf=f; best=t
        th[c]=best
        print(c,best)
    return th

# ---------- PREDICT ----------
def predict(model,tk,th):
    dev=load_dev(); model.eval()
    for lang,df in dev.items():
        preds=[]
        for txt in tqdm(df.text,desc=lang):
            enc=tk(txt,return_tensors="pt",truncation=True,max_length=MAX_LEN).to(DEVICE)
            cid=torch.tensor([CLUSTER2ID[LANG2CLUSTER[lang]]]).to(DEVICE)
            logits,_=model(enc["input_ids"],enc["attention_mask"],cid)
            prob=torch.softmax(logits,1)[0,1].item()
            preds.append(int(prob>th[LANG2CLUSTER[lang]]))
        pd.DataFrame({"id":df.id,"polarization":preds}).to_csv(f"{PRED_DIR}/pred_{lang}.csv",index=False)

# ---------- RUN ----------
model,tk,val=train()
val=normalize_probs(val)
th=tune_cluster_thresholds(val)
predict(model,tk,th)
